In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops
import os, glob
import cv2

In [5]:
DATASET_DIR = "datasets"
IMG_SIZE = (256, 256)
LEVELS = 64
DISTANCE = [1, 2, 3]
ANGLES = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

In [14]:
rows = []

for dataset in os.listdir(DATASET_DIR):
    dataset_path = os.path.join(DATASET_DIR, dataset)
    if not os.path.isdir(dataset_path):
        continue

    img_files = []
    for extension in ("*.png", "*.jpg", "*.jpeg"):
        img_files.extend(glob.glob(os.path.join(dataset_path, extension)))

    for img_file in img_files:
        img = cv2.imread(img_file, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Gagal membaca gambar: {img_file}")

        img_resized = cv2.resize(img, IMG_SIZE)
        img_quantized = (img_resized.astype(np.uint16) * LEVELS // 256).astype(np.uint8)

        glcm = graycomatrix(
            img_quantized,
            distances=DISTANCE,
            angles=ANGLES,
            levels=LEVELS,
            symmetric=True,
            normed=True,
        )

        for prop in PROPS:
            for value in graycoprops(glcm, prop).flatten():
                rows.append([dataset, img_file, prop, value])

df = pd.DataFrame(rows, columns=["dataset", "image", "property", "value"])
if df.empty:
    raise ValueError(f"Tidak ada gambar yang ditemukan di {DATASET_DIR}")

df.to_csv(
    "glcm_features.csv",
    mode="a",
    header=not os.path.exists("glcm_features.csv"),
    index=False,
)

print(df.shape)
print(df["dataset"].value_counts())


(10872, 4)
dataset
Yosia    3672
Felix    3600
Eric     3600
Name: count, dtype: int64
